# Kaira君検出 YOLO の学習（Google Colab）

`app/`（Kaira君を探せ）のヒントに使う YOLO を学習する。

- データ：Google ドライブの `ai_worry/dataset/`（YOLO 形式）
  ```
  dataset/
    images/train, images/val   画像
    labels/train, labels/val   画像と同じ名前の .txt（1行 = `クラス番号 中心x 中心y 幅 高さ`、0〜1 に正規化）
  ```
- 出力：`ai_worry/models/yolo26n_kaira.pt`。リポジトリの `data/best.pt` に置くと、アプリが YOLO でヒントを出す

メニューの「ランタイム → ランタイムのタイプを変更」で GPU（T4 など）を選んでから、上から順に実行する。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!pip install -q -U ultralytics

## 設定

In [ ]:
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/ai_worry")
DATASET_DIR = DRIVE_DIR / "dataset"
OUTPUT_PATH = DRIVE_DIR / "models" / "yolo26n_kaira.pt"
# ドライブから読むと遅いので、学習の前にローカルへ写す
LOCAL_DATASET_DIR = Path("/content/dataset")

BASE_MODEL = "yolo26n.pt"  # 事前学習済みの重み（自動でダウンロードされる）
EPOCHS = 100
# 問題画像は 1024px で、Kaira君は小さく写るので、縮めすぎないよう 1024 で学習する（メモリが足りなければ 640 に）
IMAGE_SIZE = 1024
BATCH = 16
PATIENCE = 30  # この回数のあいだ val の成績が上がらなければ早めに止める

## データの確認

In [ ]:
from collections import Counter

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
class_counts = Counter()
for split in ("train", "val"):
    images = sorted(p for p in (DATASET_DIR / "images" / split).iterdir() if p.suffix.lower() in IMAGE_SUFFIXES)
    labels = {p.stem: p for p in (DATASET_DIR / "labels" / split).glob("*.txt")}
    missing = [p.name for p in images if p.stem not in labels]
    boxes = 0
    for p in labels.values():
        for line in p.read_text().splitlines():
            if line.strip():
                class_counts[int(line.split()[0])] += 1
                boxes += 1
    print(f"{split}: 画像 {len(images)} 枚、ラベル {len(labels)} 個、枠 {boxes} 個、ラベルのない画像 {len(missing)} 枚 {missing[:5]}")
print("クラスごとの枠の数:", dict(sorted(class_counts.items())))

## クラス名

ラベルのクラス番号に合わせて名前を付ける。アプリはクラス名に `real` / `fake` / `kaira` のどれかが入っていれば Kaira君用のモデルとみなす。
番号の意味がラベル作成時と違う場合は `CLASS_NAMES` を直す。

In [ ]:
num_classes = max(class_counts) + 1
CLASS_NAMES = {1: ["kaira"], 2: ["real", "fake"]}.get(num_classes, [f"kaira_{i}" for i in range(num_classes)])
print(CLASS_NAMES)

In [ ]:
import shutil

import yaml

# ローカルへ写す（前回の写しとキャッシュは消す）
shutil.rmtree(LOCAL_DATASET_DIR, ignore_errors=True)
shutil.copytree(DATASET_DIR, LOCAL_DATASET_DIR, ignore=shutil.ignore_patterns("*.cache"))

data_yaml = Path("/content/kaira.yaml")
data_yaml.write_text(yaml.safe_dump({
    "path": str(LOCAL_DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": dict(enumerate(CLASS_NAMES)),
}, allow_unicode=True))
print(data_yaml.read_text())

## 学習

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    project="/content/runs",
    name="kaira",
    exist_ok=True,
    # 問題画像ではキャラを左右反転して置くので、左右反転の水増しは使う。上下反転・大きな回転は使わない
    fliplr=0.5,
    flipud=0.0,
    degrees=10.0,
)

## 評価

In [ ]:
best = YOLO("/content/runs/kaira/weights/best.pt")
metrics = best.val(data=str(data_yaml), imgsz=IMAGE_SIZE)
print(f"mAP50: {metrics.box.map50:.3f}  mAP50-95: {metrics.box.map:.3f}")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: precision {metrics.box.p[i]:.3f}  recall {metrics.box.r[i]:.3f}")

In [ ]:
# val の画像に検出結果を描いて見る
from IPython.display import Image as Show, display

val_images = sorted((LOCAL_DATASET_DIR / "images" / "val").iterdir())[:4]
for result in best.predict([str(p) for p in val_images], imgsz=IMAGE_SIZE, conf=0.25, save=True, project="/content/runs", name="predict", exist_ok=True):
    print(Path(result.path).name, len(result.boxes), "個")
for p in sorted(Path("/content/runs/predict").iterdir())[:4]:
    display(Show(filename=str(p), width=600))

## 保存

ドライブの `ai_worry/models/yolo26n_kaira.pt` に保存する。
アプリで使うときは、これをリポジトリの `data/best.pt` として置き、`app/README.md` の手順で ultralytics を入れて起動する。

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
shutil.copy("/content/runs/kaira/weights/best.pt", OUTPUT_PATH)
print("saved", OUTPUT_PATH)